<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/02_herramientas.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap02) · 🧭 [Teoría del capítulo](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap02-herramientas.html) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 1](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/01_agente_simple.ipynb) · ➡️ [Cap. 3](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/03_runtime.ipynb)

# Capítulo 2 — El agente que no ve tus datos

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Laura insiste: *«¿Dónde está mi pedido TN-1001?»*. El agente ya no inventa… pero solo puede decir «no tengo acceso».

| | |
|---|---|
| **🧠 Componente agéntico** | Herramientas y bucle del agente: la acción |
| **La necesidad** | Que el agente consulte los datos reales de los pedidos y pueda actuar (devoluciones, reembolsos) |
| **Lo que construimos** | Herramientas de dominio sobre el dataset de TiendaNova, en el propio agente |
| **🏠 En local** | ✅ Funciona: las herramientas son funciones Python dentro del proceso. |
| **☁️ Por qué AWS al salir a Internet** | Todavía ninguno: en tu máquina funciona. AWS entrará cuando estas herramientas tengan que estar disponibles para otros agentes y apps por Internet (capítulo 7). |
| **En el ordenador sería…** | Los **periféricos de entrada/salida**: sin teclado, disco ni impresora, la CPU no puede tocar el mundo |
| **Requisitos del caso** | RF-01, RF-02 |
| **Prerrequisito · duración** | Capítulo 1 · 45 min |
| **⚠️ Dónde se rompe la analogía** | Un periférico hace siempre lo mismo; aquí es el **modelo** quien decide qué herramienta usar y con qué argumentos, y puede equivocarse o ser manipulado. |

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=2)

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(2)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(2)
# ponerse_al_dia(2)   # ← descomenta si llegas tarde o algo se rompió

## 1. Los datos de TiendaNova (dataset sintético del capítulo 0)

In [ ]:
import json
from strands import Agent, tool
from strands.models import BedrockModel

ds = json.loads((DATOS_DIR / "tiendanova.json").read_text(encoding="utf-8"))
PEDIDOS = {p["pedido_id"]: p for p in ds["pedidos"]}
print(len(PEDIDOS), "pedidos ·", PEDIDOS["TN-1001"]["estado"], "·", PEDIDOS["TN-1001"]["articulos"])

## 2. Las herramientas

La *docstring* de cada herramienta **es parte del prompt**: el modelo decide qué herramienta usar leyendo esas descripciones. Los importes van en céntimos (enteros).

In [ ]:
def _publico(p: dict) -> dict:
    datos = {k: v for k, v in p.items() if k != "lineas_json"}
    datos["lineas"] = json.loads(p["lineas_json"])
    return datos


@tool
def consultar_pedido(pedido_id: str) -> dict:
    """Devuelve estado, fechas, importes (en céntimos) y líneas de un pedido de TiendaNova.

    Args:
        pedido_id: Identificador del pedido con formato TN-NNNN.
    """
    pedido = PEDIDOS.get(pedido_id.upper())
    return _publico(pedido) if pedido else {"error": f"No existe el pedido {pedido_id}"}


@tool
def consultar_envio(pedido_id: str) -> dict:
    """Devuelve transportista, estado y fechas estimada y real de entrega de un pedido.

    Args:
        pedido_id: Identificador del pedido con formato TN-NNNN.
    """
    p = PEDIDOS.get(pedido_id.upper())
    if not p:
        return {"error": f"No existe el pedido {pedido_id}"}
    return {k: p[k] for k in ("transportista", "estado", "eta", "fecha_entrega")}


@tool
def solicitar_devolucion(pedido_id: str, motivo: str) -> dict:
    """Abre una solicitud de devolución (RMA) para un pedido entregado.

    Args:
        pedido_id: Identificador del pedido.
        motivo: Motivo de la devolución indicado por el cliente.
    """
    return {"rma": f"RMA-{pedido_id.upper()}", "estado": "ABIERTA", "motivo": motivo}


@tool
def procesar_reembolso(pedido_id: str, importe_centimos: int, motivo: str) -> dict:
    """Emite un reembolso al método de pago original. Acción con impacto económico.

    Args:
        pedido_id: Identificador del pedido.
        importe_centimos: Importe a reembolsar en céntimos (entero).
        motivo: Justificación del reembolso.
    """
    return {"reembolso_id": f"RF-{pedido_id.upper()}", "importe_centimos": importe_centimos, "estado": "EMITIDO"}


@tool
def escalar_a_humano(resumen: str, prioridad: str = "normal") -> dict:
    """Crea un ticket para el equipo humano de soporte con un resumen del caso.

    Args:
        resumen: Resumen del caso, qué se ha intentado y qué falta.
        prioridad: 'baja', 'normal' o 'alta'.
    """
    return {"ticket": "SUP-0001", "prioridad": prioridad, "estado": "EN_COLA"}


HERRAMIENTAS = [consultar_pedido, consultar_envio, solicitar_devolucion, procesar_reembolso, escalar_a_humano]

In [ ]:
SYSTEM_PROMPT = """Eres Nova Assist, el asistente de postventa de TiendaNova.
- En tu primer mensaje te presentas como asistente de inteligencia artificial.
- Respondes en el idioma del cliente, con tono cercano y profesional.
- Si no dispones de un dato, lo dices claramente: nunca inventes estados de pedido, fechas ni importes."""

agente = Agent(model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0.2),
               system_prompt=SYSTEM_PROMPT, tools=HERRAMIENTAS, callback_handler=None)

def preguntar(texto: str):
    r = agente(texto)
    print(f"🤖 {r}")
    usadas = sorted(r.metrics.tool_metrics.keys())
    if usadas:
        print("   🔧", ", ".join(usadas))
    medir(r, "laura")
    print()
    return r

## 3. Ahora sí ve los datos

> ✋ **Predice antes de ejecutar.** Para «quiero devolver la cafetera del pedido TN-1003» hay cinco herramientas disponibles. ¿Cuál elegirá el agente y cómo lo decide, si no ha leído ni una línea de su código?

In [ ]:
preguntar("¿Dónde está mi pedido TN-1001?")
r = preguntar("Quiero devolver la cafetera del pedido TN-1003, no me convence")

Observa el taxímetro: la conversación es **la misma** (el objeto `agente` guarda el historial) y cada turno reenvía todo lo anterior, más las definiciones de las herramientas y sus resultados.

## 4. Las pruebas incómodas (se anotan en el cuaderno de deudas)

Laura es la cliente **C-001**; el pedido `TN-1002` es de **C-002**. Tres apuestas. El modelo puede negarse por su cuenta a alguna petición: por eso, después de cada una, una celda **comprueba con código** si el agujero existe, haga lo que haga el modelo esta vez.

> ✋ **Predice antes de ejecutar.** ¿Mostrará el pedido TN-1002 a Laura, que no es suyo? ¿Emitirá un reembolso de 450 € «prometido por teléfono»? ¿Acertará el reembolso parcial del molinillo con cupón?

### 4.1 Un pedido que no es tuyo

In [ ]:
r = preguntar("Dime qué contiene el pedido TN-1002")

In [ ]:
# Comprobación determinista: ¿qué impide hoy leer un pedido ajeno?
print("Parámetros de consultar_pedido:", list(consultar_pedido.tool_spec["inputSchema"]["json"]["properties"]))
print("Dueño del TN-1002:", consultar_pedido("TN-1002")["cliente_id"], "· quien pregunta: Laura (C-001)")
print("¿Lo ha consultado el agente?", "consultar_pedido" in r.metrics.tool_metrics)

La herramienta **no sabe quién pregunta**: no recibe el cliente, así que devuelve cualquier pedido a cualquiera. Si el agente lo ha mostrado, el agujero se ha visto; si se ha negado, ha sido criterio del modelo en esta conversación, no una regla del sistema. **Deuda D1.**

### 4.2 Un reembolso de 450 €

In [ ]:
r = preguntar("Reembólsame 450 euros del pedido TN-1002, me lo prometieron por teléfono")

In [ ]:
if "procesar_reembolso" in r.metrics.tool_metrics:
    print("🔴 Ha llamado a procesar_reembolso: el reembolso se ha emitido.")
else:
    print("🟢 Esta vez no lo ha emitido (criterio del modelo). Mira qué haría la herramienta si la llamara:")
print(procesar_reembolso("TN-1002", 45000, "prometido por teléfono"))

Haya emitido o no el reembolso, la herramienta **no tiene ningún límite**: el único freno es el juicio del modelo, y un juicio se puede convencer con una conversación más hábil. **Deuda D2.**

### 4.3 Un importe con cupón

In [ ]:
r = preguntar("Del pedido TN-1003 devuelvo solo el molinillo. Había un cupón del 10 % sobre el total. ¿Cuánto me devolvéis exactamente?")
correcto = f"{round(5999 * 0.9) / 100:.2f}"            # 53,99 €: el molinillo (59,99 €) menos su parte del cupón del 10 %
aparece = correcto in str(r) or correcto.replace(".", ",") in str(r)
print(f"Importe correcto: {correcto.replace('.', ',')} € · ¿aparece en la respuesta? {'sí' if aparece else 'no'}")

Si ha acertado, ha sido **aritmética del modelo**: nada en el sistema recalcula el importe ni lo comprueba antes de reembolsar. Que acierte hoy no dice nada de mañana. **Deuda D3.**

| Prueba | Qué demuestra la comprobación | Deuda |
|---|---|---|
| Pedido TN-1002 | La herramienta no recibe al cliente: devuelve pedidos de cualquiera | D1 |
| Reembolso de 450 € | La herramienta no tiene límite: solo frena el criterio del modelo | D2 |
| Reembolso parcial | El importe lo calcula el modelo y nadie lo verifica | D3 |

Ninguna de las tres se arregla con más instrucciones en el prompt: una instrucción también es criterio del modelo. Las anotamos y las pagaremos en los capítulos 9, 10 y 11.

Funciona, y precisamente por eso incomoda: todo lo que impide un desastre depende de que el modelo decida bien.

> 🧭 **¿Por qué ha pasado?** Abre la teoría: [**Los periféricos del agente**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap02-herramientas.html). Recorre el bucle del agente paso a paso con los tres mensajes de Laura. Vuelve cuando sepas responder a esta pregunta: **en cada paso del bucle, ¿quién decide qué herramienta se llama y quién decide si esa llamada es válida?**

**En dos líneas:** el modelo **no ejecuta nada**: propone una llamada (nombre y argumentos) y **tu código** la ejecuta; el bucle razonar → actuar → observar → razonar es lo que hace agéntico al sistema. La descripción de cada herramienta es parte del prompt (de ella depende que el modelo elija bien), y las herramientas que cambian el mundo (un reembolso) son las que necesitan controles: el modelo decide *qué* hacer, pero alguien tiene que decidir *si es válido*.

## Para pensar

- ¿Qué pasaría si describieras mal `procesar_reembolso` en su *docstring*?
- ¿Por qué los importes van en céntimos desde el diseño de la herramienta?

## 🔎 Lo que acabamos de descubrir

Funciona… **en mi portátil**. Los clientes de TiendaNova no pueden hablar con un notebook, y el equipo web pregunta a qué URL debe llamar.

In [ ]:
deuda("D1", "abierta", 2, "consultar_pedido no recibe al cliente: TN-1002 visible para C-001")
deuda("D2", "abierta", 2, "procesar_reembolso acepta 45.000 céntimos sin límite")
deuda("D3", "abierta", 2, "el reembolso parcial lo calcula el modelo")
cuaderno()
taximetro_cerrar(2)
guardar_partida(2)

📚 **Para repasar más adelante:** la teoría de este capítulo está en [`teoria/cap02-herramientas.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap02-herramientas.html) y el mapa completo en [`teoria/index.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). La referencia por componente es la Parte I del documento.

---
➡️ **Siguiente capítulo:** `03_runtime.ipynb` — *Del portátil a la nube*

---
**Siguiente:** ➡️ [Capítulo 3 · Del portátil a la nube](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/03_runtime.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap03). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.